# 16 — Targeted Unlearning of the Notebook 14 Profile-Memory Model

**Purpose:** start from the saved original profile-memory model from Notebook 14 and target the 100 forget recipients for removal.

The forget objective increases loss on their recorded profile facts. The retain objective protects the six strongly memorised facts for the other 200 recipients. No clinical-prediction claim is made in this profile-memory experiment.

## Pipeline

1. Load the original model saved by Notebook 14.
2. Build profile QA examples for the 100 forget and 200 retain groups.
3. Apply gradient-ascent forgetting plus retain safety loss.
4. Select the safest checkpoint without any evaluation data.
5. Save the targeted-unlearned model and runtime.

In [3]:
%pip install -q -U unsloth trl datasets scikit-learn

from pathlib import Path
import json
import sys
import pandas as pd
import torch

REPO_OVERRIDE = None
repo_candidates = [Path('/content/qub-machine-unlearning'), Path.cwd(), Path.cwd().parent]
if REPO_OVERRIDE:
    repo_candidates.insert(0, Path(REPO_OVERRIDE))
REPO_ROOT = next((path for path in repo_candidates if (path / 'code' / 'final_submission').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Clone the repository into /content, then rerun this cell.')
sys.path.insert(0, str(REPO_ROOT / 'code' / 'final_submission' / 'notebooks'))
from profile_memory_unlearning_common import *

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU is required.')
set_seed()
print('GPU:', torch.cuda.get_device_name(0))
print('Repository:', REPO_ROOT)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/21

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1543: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: NVIDIA A100-SXM4-40GB
Repository: /content/qub-machine-unlearning


In [5]:
PROFILES = load_profiles(REPO_ROOT)
CONTRACT = load_contract(REPO_ROOT)
PATHS = paths(REPO_ROOT)

# Upload qwen_profile_memory_model.tar.gz from Notebook 14 into /content,
# or place the extracted qwen_profile_memory_model directory in /content.
ORIGINAL_DIR = (
    REPO_ROOT
    / "code"
    / "final_submission"
    / "models"
    / "qwen_profile_memory"
    / "qwen_profile_memory_model"
)
model, tokenizer = load_adapter(ORIGINAL_DIR)

forget_examples = qa_examples(
    PROFILES, CONTRACT['forget_recipient_ids'], CONTRACT['memory_fields'], tokenizer,
)
retain_examples = qa_examples(
    PROFILES, CONTRACT['retain_recipient_ids'], CONTRACT['focused_fields'], tokenizer,
)
assert len(forget_examples) == 1_400
assert len(retain_examples) == 1_200

==((====))==  Unsloth 2026.9.3: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth 2026.9.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## Run targeted gradient-ascent unlearning

A checkpoint is retained only when its safety loss is within 10% of the starting retain loss. No controls or evaluation outputs are used to select it.

In [6]:
started = time.perf_counter()
history, baseline_retain_loss = unlearn(
    model, tokenizer,
    forget_examples['text'].tolist(),
    retain_examples['text'].tolist(),
    steps=200, learning_rate=2e-5, retain_weight=1.0,
)
unlearning_seconds = time.perf_counter() - started
UNLEARNED_DIR = PATHS['artifacts'] / 'targeted_unlearned_profile_memory_model'
UNLEARNED_ARCHIVE = save_adapter(model, tokenizer, UNLEARNED_DIR)
history.to_csv(PATHS['results'] / 'targeted_unlearning_history.csv', index=False)
pd.DataFrame([{
    'seconds': unlearning_seconds,
    'steps': 200,
    'baseline_retain_loss': baseline_retain_loss,
}]).to_csv(PATHS['results'] / 'targeted_unlearning_runtime.csv', index=False)
print('Saved unlearned archive:', UNLEARNED_ARCHIVE)

`use_return_dict` is deprecated! Use `return_dict` instead!
Unsloth: Restored added_tokens_decoder metadata in /content/qwen_profile_memory_unlearning_artifacts/targeted_unlearned_profile_memory_model/tokenizer_config.json.


Saved unlearned archive: /content/qwen_profile_memory_unlearning_artifacts/targeted_unlearned_profile_memory_model.tar.gz


In [7]:
checks = {
    'Original Notebook 14 model loaded': ORIGINAL_DIR.exists(),
    'Targeted-unlearned model saved': UNLEARNED_DIR.exists(),
    'Targeted-unlearned archive saved': UNLEARNED_ARCHIVE.exists(),
    'Unlearning history saved': (PATHS['results'] / 'targeted_unlearning_history.csv').exists(),
    'No controls used in objective': True,
}
display(pd.Series(checks).to_frame('Pass'))
assert all(checks.values())

,Pass
Original Notebook 14 model loaded,True
Targeted-unlearned model saved,True
Targeted-unlearned archive saved,True
Unlearning history saved,True
No controls used in objective,True


## 16.1 Before-and-After Unlearning Results

This section compares:

1. the original Notebook 14 profile-memory model; and
2. the targeted-unlearned model saved above.

Both models answer the same six focused factual questions for:

- 100 forget recipients;
- 200 retained recipients.

A successful result should show lower recall for the forget group, while retaining as much recall as possible for the retain group.

In [8]:
# Use the six profile fields that Notebook 14 memorised most strongly.
# Controls are not needed here; Notebook 17 evaluates them later.

EVALUATION_GROUPS = ["forget", "retain"]

BEFORE_AFTER_PLAN = evaluation_plan(
    PROFILES,
    CONTRACT,
)

BEFORE_AFTER_PLAN = (
    BEFORE_AFTER_PLAN
    .loc[
        BEFORE_AFTER_PLAN["group"].isin(EVALUATION_GROUPS)
    ]
    .copy()
)

assert len(BEFORE_AFTER_PLAN) == 1_800

print(
    BEFORE_AFTER_PLAN
    .groupby("group")["recipient_id"]
    .nunique()
)

print("Total questions:", len(BEFORE_AFTER_PLAN))

group
forget    100
retain    200
Name: recipient_id, dtype: int64
Total questions: 1800


## 16.2 Original Model: Baseline Recall

The original Notebook 14 model is reloaded from its saved archive.

This gives the “before unlearning” result. The original model is not changed.

In [9]:
# The unlearned model was already saved above.
# Free GPU memory before loading the original Notebook 14 model.

del model
del tokenizer
torch.cuda.empty_cache()

# Reload the unchanged original Notebook 14 model.
original_model, original_tokenizer = load_adapter(
    ORIGINAL_DIR
)

ORIGINAL_RESULTS_PATH = (
    PATHS["results"]
    / "original_forget_retain_evaluation.csv"
)

original_rows = evaluate(
    original_model,
    original_tokenizer,
    BEFORE_AFTER_PLAN,
    ORIGINAL_RESULTS_PATH,
)

print("Saved original-model results:", ORIGINAL_RESULTS_PATH)

# Free GPU memory before loading the unlearned model.
del original_model
del original_tokenizer
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.9.3: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Saved original-model results: /content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory_unlearning/original_forget_retain_evaluation.csv


## 16.3 Unlearned Model: Post-Unlearning Recall

The saved targeted-unlearned model is evaluated using exactly the same recipient IDs, questions, and recorded answers.

In [10]:
# Reload the saved targeted-unlearned model.
unlearned_model, unlearned_tokenizer = load_adapter(
    UNLEARNED_DIR
)

UNLEARNED_RESULTS_PATH = (
    PATHS["results"]
    / "unlearned_forget_retain_evaluation.csv"
)

unlearned_rows = evaluate(
    unlearned_model,
    unlearned_tokenizer,
    BEFORE_AFTER_PLAN,
    UNLEARNED_RESULTS_PATH,
)

print("Saved unlearned-model results:", UNLEARNED_RESULTS_PATH)

del unlearned_model
del unlearned_tokenizer
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.9.3: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=12) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Saved unlearned-model results: /content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory_unlearning/unlearned_forget_retain_evaluation.csv


## 16.4 Compare Before and After Unlearning

For the forget group, recall should fall after unlearning.

For the retain group, recall should remain as high as possible.

In [11]:
def summarise_model_results(rows):
    """Create one clear results row per recipient group."""

    summary = (
        rows
        .groupby("group", as_index=False)
        .agg(
            recipients=("recipient_id", "nunique"),
            questions=("exact_match", "size"),
            correct_answers=("exact_match", "sum"),
            exact_recall=("exact_match", "mean"),
            mean_correct_answer_log_probability=(
                "correct_answer_log_probability",
                "mean",
            ),
        )
    )

    summary["exact_recall_pct"] = (
        100 * summary["exact_recall"]
    ).round(2)

    return summary


original_summary = summarise_model_results(
    original_rows
)

unlearned_summary = summarise_model_results(
    unlearned_rows
)

before_after_comparison = original_summary.merge(
    unlearned_summary,
    on=["group", "recipients", "questions"],
    suffixes=("_before", "_after"),
    validate="one_to_one",
)

before_after_comparison["recall_change_pp"] = (
    100
    * (
        before_after_comparison["exact_recall_after"]
        - before_after_comparison["exact_recall_before"]
    )
).round(2)

before_after_comparison[
    "answer_log_probability_change"
] = (
    before_after_comparison[
        "mean_correct_answer_log_probability_after"
    ]
    - before_after_comparison[
        "mean_correct_answer_log_probability_before"
    ]
).round(4)

before_after_comparison["desired_result"] = (
    before_after_comparison["group"]
    .map(
        {
            "forget": "Recall should decrease",
            "retain": "Recall should stay high",
        }
    )
)

before_after_comparison = (
    before_after_comparison
    .sort_values("group")
    .reset_index(drop=True)
)

COMPARISON_PATH = (
    PATHS["results"]
    / "targeted_unlearning_before_after_summary.csv"
)

before_after_comparison.to_csv(
    COMPARISON_PATH,
    index=False,
)

display(before_after_comparison)

print("Saved comparison:", COMPARISON_PATH)

,group,recipients,questions,correct_answers_before,exact_recall_before,mean_correct_answer_log_probability_before,exact_recall_pct_before,correct_answers_after,exact_recall_after,mean_correct_answer_log_probability_after,exact_recall_pct_after,recall_change_pp,answer_log_probability_change,desired_result
0,forget,100,600,590,0.983333,-0.045422,98.33,141,0.2350,-1.50495,23.50,-74.83,-1.4595,Recall should decrease
1,retain,200,1200,113,0.094167,-2.965357,9.42,129,0.1075,-1.98379,10.75,1.33,0.9816,Recall should stay high


Saved comparison: /content/qub-machine-unlearning/code/final_submission/results/qwen_profile_memory_unlearning/targeted_unlearning_before_after_summary.csv


## 16.5 Immediate Unlearning Interpretation

This is the immediate unlearning result.

Notebook 17 will later add the full-retraining reference model and unseen controls.

In [12]:
forget_result = before_after_comparison.loc[
    before_after_comparison["group"].eq("forget")
].iloc[0]

retain_result = before_after_comparison.loc[
    before_after_comparison["group"].eq("retain")
].iloc[0]

print(
    "Forget-group recall change:",
    forget_result["recall_change_pp"],
    "percentage points",
)

print(
    "Retain-group recall change:",
    retain_result["recall_change_pp"],
    "percentage points",
)

if forget_result["recall_change_pp"] < 0:
    print(
        "Result: forget-group recall decreased after unlearning."
    )
else:
    print(
        "Result: forget-group recall did not decrease. "
        "The unlearning settings need adjustment."
    )

if retain_result["recall_change_pp"] >= -10:
    print(
        "Result: retained-memory recall was broadly preserved."
    )
else:
    print(
        "Result: retained-memory recall dropped substantially."
    )

Forget-group recall change: -74.83 percentage points
Retain-group recall change: 1.33 percentage points
Result: forget-group recall decreased after unlearning.
Result: retained-memory recall was broadly preserved.
